# Task 1 – LoRA fine-tuning: tweet → emotion

Qwen2.5-0.5B-Instruct (494M parameters) learns to label a tweet as `sadness, joy, love, anger, fear` or `surprise`.
**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (roughly 10 minutes). If Colab asks for a restart after the first code cell, restart and run all again.

**Metric:** accuracy = the model's answer starts with the correct label, on 300 held-out tweets (greedy decoding). Base and fine-tuned model get the identical prompt, so the difference is due to fine-tuning only.

| The task asks for | Where (code cells are numbered `# 1` to `# 5`) |
|---|---|
| Base vs fine-tuned on ≥ 50 held-out examples | cell 3 (base) and cell 5 (fine-tuned) |
| LoRA settings and why | values in cell 1, reasons in the table above cell 4 |
| 3 failures and why | cell 5 |
| 7B memory: full / LoRA / QLoRA | last section |

In [1]:
# 1 - setup and settings
!pip install -q -U transformers peft accelerate datasets

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"      # also fine: "Qwen/Qwen2.5-1.5B-Instruct" (set BATCH = 8) or "HuggingFaceTB/SmolLM2-360M-Instruct"
N_TRAIN, N_TEST = 3000, 300               # training tweets, held-out test tweets
RANK, ALPHA, DROPOUT = 16, 32, 0.05       # LoRA settings (reasons below)
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LR, EPOCHS, BATCH = 2e-4, 2, 16

device = "cuda" if torch.cuda.is_available() else "cpu"   # shortcut: CPU runs but takes hours; use the T4 runtime
print("device:", device)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
device: cuda


In [2]:
# 2 - data and prompt
ds = load_dataset("dair-ai/emotion")                       # 16k tweets, 6 labels
names = ds["train"].features["label"].names                # ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
train = ds["train"].shuffle(seed=0).select(range(N_TRAIN))
seen = set(train["text"])
test = ds["test"].shuffle(seed=0).select(range(N_TEST)).filter(lambda r: r["text"] not in seen)   # official test split, minus any duplicate of a training text

tok = AutoTokenizer.from_pretrained(MODEL)

def make_prompt(text):                                     # one prompt for the base AND the fine-tuned model
    msgs = [{"role": "system", "content": "Classify the emotion of the text. Answer with one word: " + ", ".join(names) + "."},
            {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(len(train), "train /", len(test), "held-out test\n")
print(make_prompt(train[0]["text"]), "->", names[train[0]["label"]])

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

... (7 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

3000 train / 300 held-out test

<|im_start|>system
Classify the emotion of the text. Answer with one word: sadness, joy, love, anger, fear, surprise.<|im_end|>
<|im_start|>user
i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is more mature<|im_end|>
<|im_start|>assistant
 -> joy


In [3]:
# 3 - BASE model: accuracy before fine-tuning
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)    # fp32: 0.5B x 4 bytes = 2 GB, no precision surprises

@torch.no_grad()
def predict(model, text):
    x = tok(make_prompt(text), return_tensors="pt", add_special_tokens=False).to(device)
    out = model.generate(**x, max_new_tokens=6, do_sample=False, repetition_penalty=1.0)   # 1.0: the model's default 1.1 would penalise the label words
    return tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()

def evaluate(model):
    preds = [predict(model, r["text"]) for r in test]
    return preds, sum(p.startswith(names[r["label"]]) for p, r in zip(preds, test)) / len(test)

base_preds, base_acc = evaluate(model)
print(f"BASE accuracy: {base_acc:.1%}   first outputs: {base_preds[:6]}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

BASE accuracy: 28.7%   first outputs: ['anger', 'sadness', 'love', 'sadness', 'love', 'anger']


## LoRA settings and why
LoRA freezes every original weight and learns a small correction next to each chosen matrix: `W·x + (alpha/r)·B·A·x`, with `A` of size `r × d_in` and `B` of size `d_out × r`. Only `A` and `B` are trained (about 1.7% of the parameters here). These are standard defaults, not tuned.

| Setting | Value | Why |
|---|---|---|
| rank `r` | 16 | The update has rank at most `r`. A narrow task needs little new capacity, 8–16 is the usual range, and the QLoRA paper found `r` matters little once all linear layers are adapted. Larger `r` mainly costs memory. |
| `alpha` | 32 | The correction is scaled by `alpha/r` = 2. `alpha = 2r` is a common default that stays stable at the learning rate below. |
| target modules | `q,k,v,o` + `gate,up,down` | All linear layers. The QLoRA paper found that adapting only `q,v` is clearly weaker, and it is cheap on a small model. |
| dropout | 0.05 | Mild regularisation; the labels are noisy. |
| learning rate | 2e-4, cosine decay, 20 warm-up steps | The LoRA update starts at zero and the adapters are tiny, so they work best at roughly 10× the learning rate of full fine-tuning (≈ 1e-5). 1e-4 to 3e-4 is the standard range. |
| epochs / batch | 2 / 16 (≈ 375 steps) | 3,000 examples with noisy labels: two passes are enough, more mostly memorise the noise. |

In [5]:
!pip uninstall -y -q torchao

In [6]:
!pip install -q -U transformers peft accelerate datasets
!pip uninstall -y -q torchao

In [7]:
# 4 - LoRA fine-tuning
model = get_peft_model(model, LoraConfig(r=RANK, lora_alpha=ALPHA, lora_dropout=DROPOUT, target_modules=TARGETS, task_type="CAUSAL_LM"))
model.print_trainable_parameters()

def encode(r):                                             # loss only on the answer: prompt tokens get label -100 (ignored)
    p = tok(make_prompt(r["text"]), add_special_tokens=False)["input_ids"]
    a = tok(names[r["label"]] + tok.eos_token, add_special_tokens=False)["input_ids"]   # eos teaches the model to stop
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

ex = encode(train[0])                                      # self-check: the only tokens with loss are "<label><eos>"
assert tok.decode([t for t in ex["labels"] if t != -100]) == names[train[0]["label"]] + tok.eos_token

bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8                  # a T4 has no bf16, so it uses fp16
# shortcut: fp16 can overflow on some Qwen layers; if the training loss prints nan, set fp16=False below (slower, still fits)
Trainer(model=model, train_dataset=[encode(r) for r in train], data_collator=DataCollatorForSeq2Seq(tok, padding=True),
        args=TrainingArguments(output_dir="out", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH, learning_rate=LR,
                               lr_scheduler_type="cosine", warmup_steps=20, logging_steps=20, save_strategy="no",
                               report_to="none", bf16=bf16, fp16=device == "cuda" and not bf16)).train()

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
20,0.637089
40,0.472514
60,0.290480
80,0.259929
100,0.348591
120,0.232451
140,0.210285
160,0.143064
180,0.116540
200,0.136326


TrainOutput(global_step=376, training_loss=0.18624956747318835, metrics={'train_runtime': 194.4576, 'train_samples_per_second': 30.855, 'train_steps_per_second': 1.934, 'total_flos': 1111705899417600.0, 'train_loss': 0.18624956747318835, 'epoch': 2.0})

In [8]:
# 5 - FINE-TUNED model: accuracy on the same test set, then its mistakes
ft_preds, ft_acc = evaluate(model)
print(f"accuracy on {len(test)} held-out tweets:  base {base_acc:.1%}  ->  fine-tuned {ft_acc:.1%}")

wrong = [(r["text"], names[r["label"]], p) for p, r in zip(ft_preds, test) if not p.startswith(names[r["label"]])]
print(f"\n{len(wrong)} still wrong. First 10 (pick 3 with different causes):")
for text, true, pred in wrong[:10]:
    print(f"- {text}\n    true: {true} | model said: {pred!r}")

accuracy on 300 held-out tweets:  base 28.7%  ->  fine-tuned 89.7%

31 still wrong. First 10 (pick 3 with different causes):
- i do feel stressed
    true: sadness | model said: 'anger'
- i hope it is because he understands the way i feel i hope he sees what he could miss and is putting the petty negative thoughts aside
    true: anger | model said: 'sadness'
- i am right handed however i play billiards left handed naturally so me trying to play right handed feels weird
    true: surprise | model said: 'fear'
- i hate these feelings in my heart i hate that work stressed me out i hate that cornelius wont let me get my way im frustrated lord
    true: sadness | model said: 'anger'
- i feel my heart is tortured by what i have done
    true: anger | model said: 'fear'
- i really dont like quinn because i feel like she will just end up hurting barney and i hated the lame ted robin storyline
    true: anger | model said: 'sadness'
- i just can t feel accepted
    true: love | model said: 'jo

**Why a failure happens: things to check in your own examples.** The tweet expresses two emotions and the label is only one; neighbouring classes overlap (`love`/`joy`, `fear`/`surprise`); the labels were assigned automatically from hashtags and keywords, so some are noisy or wrong; negation or sarcasm (`not happy`). LoRA with 3,000 examples teaches the task format and the label boundaries seen in the data, not new language understanding.

## Memory to fine-tune a 7B model (rough, 1 GB = 10⁹ bytes)
Assumptions: AdamW, LoRA `r = 16` on all linear layers, a Llama-7B shape (4096 hidden, 11008 MLP, 32 layers), batch 4 × 512 tokens, gradient checkpointing. Activations plus CUDA overhead ≈ **3 GB** (about 1.4 GB activations and logits, about 1.5 GB overhead); this part grows with batch × sequence length, and without checkpointing it grows by about 13 GB.

| Method | Weights | Gradients + optimizer | Activations + overhead | Total |
|---|---|---|---|---|
| (a) Full fine-tuning | 7e9 × 2 B = **14** | 7e9 × (2 + 4 + 4 + 4) B = **98** | 3 | **≈ 115 GB** |
| (b) LoRA | 7e9 × 2 B = **14** (frozen, bf16) | 40e6 × 16 B = **0.6** | 3 | **≈ 18 GB** |
| (c) QLoRA | 7e9 × 0.5 B = 3.5, plus ≈ 0.5 for 16-bit embeddings and quantisation constants = **4** | 40e6 × 16 B = **0.6** | 3 | **≈ 8 GB** |

* **Full:** every parameter keeps a bf16 weight (2 B), a bf16 gradient (2 B), an fp32 master copy (4 B), Adam's `m` (4 B) and `v` (4 B) = 16 bytes, so 7e9 × 16 = 112 GB before activations. It needs 2 × 80 GB GPUs or sharding.
* **LoRA:** the frozen base needs no gradients and no optimizer state, only its 2 bytes per parameter. The adapters have 32 layers × 16 × (4·(4096 + 4096) + 3·(4096 + 11008)) = 40M parameters, each with an fp32 weight, gradient, `m` and `v` = 16 bytes. This fits a 24 GB GPU.
* **QLoRA:** the same adapters, but the frozen base is stored in 4-bit NF4 (0.5 byte per parameter). This fits the free 15 GB T4.